# Silver: expeditions_members
Cleans and transforms `himalaya.bronze.expeditions_members` into `himalaya.silver.expeditions_members`

##Table Overview

In [0]:
import pyspark.sql.functions as F
from datetime import date

In [0]:
df = spark.table("himalaya.bronze.expeditions_members")

In [0]:
df.printSchema()

In [0]:
print(df.columns)

In [0]:
display(df.limit(3))

# Simplify Data

  > ## Drop irrelevant columns

In [0]:
drop_cols = [
    "membid", "residence", "occupation", "status", "deputy", "bconly",
    "nottobc", "support", "disabled", "tibetan", "mclaimed", "mdisputed",
    "msolo", "mtraverse", "mski", "mparapente", "mspeed", "mhighpt",
    "msmtdate2", "msmtdate3", "msmttime1", "msmttime2", "msmttime3",
    "mroute1", "mroute2", "mroute3", "mascent1", "mascent2", "mascent3",
    "mo2none", "mo2climb", "mo2descent", "mo2sleep", "mo2medical", "mo2note",
    "deathtime", "deathclass", "msmtbid", "hcn", "mchksum"
]

df = df.drop(*drop_cols)

In [0]:
print(df.columns)

In [0]:
display(df.limit(5))

  > ## Combine first and last name into one

In [0]:
df = df.withColumn("name",
    F.initcap(F.trim(F.regexp_replace(F.concat_ws(" ", F.col("fname"), F.col("lname")), r"\s+", " ")))
).drop("fname", "lname")

In [0]:
display(df.limit(5))

  > ## Distinct Data

In [0]:
df.select("sex").distinct().show()

In [0]:
df.select("mseason").distinct().show()

In [0]:
df.select("citizen").distinct().show()

> ##Selecting only first dual nationality

In [0]:
df = df.withColumn("citizen", F.split(F.col("citizen"), "/")[0])

In [0]:
df.select("citizen").distinct().show()

  > ## Casting Columns

In [0]:
# Date
df = df.withColumn("msmtdate1", F.to_date(F.col("msmtdate1"))) \
       .withColumn("deathdate", F.to_date(F.col("deathdate")))

In [0]:
# Year of birth
df = df.withColumn("yob", F.col("yob").cast("int"))

In [0]:
# Highest point
df = df.withColumn("mperhighpt", F.col("mperhighpt").cast("int"))

In [0]:
df.select("msmtdate1", "deathdate", "yob", "mperhighpt").printSchema()

  > ## Rename Columns

In [0]:
df = df.withColumnsRenamed({
    "myear": "year",
    "mseason": "season",
    "citizen": "nationality",
    "msuccess": "success",
    "mperhighpt": "highest_point",
    "msmtdate1": "summit_date",
    "mo2used": "oxygen_used",
    "deathdate": "death_date",
    "deathtype": "death_type",
    "deathhgtm": "death_height",
    "msmtterm": "summit_term"
})

  > ## Reorder Data

In [0]:
df = df.select(
    "expid", "peakid", "year", "season",
    "name", "sex", "yob", "nationality",
    "leader", "hired", "sherpa",
    "success", "highest_point", "summit_date", "summit_term",
    "oxygen_used",
    "death", "death_date", "death_type", "death_height",
    "ingested_at"
)

In [0]:
display(df.limit(5))

> ## Silver Transfer

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("himalaya.silver.expeditions_members")
print("✅ Written to himalaya.silver.expeditions_members")

In [0]:
display(spark.table("himalaya.silver.expeditions_members").limit(5))

# Transformations Applied

| Column | Transformation |
|---|---|
| `fname` + `lname` | Consolidated into `name` with proper case, trimmed and regex cleaned, originals dropped |
| `citizen` | Split on `/`, first nationality taken, renamed to `nationality` |
| `yob` | Cast from Double to Integer |
| `mperhighpt` | Cast from Double to Integer, renamed to `highest_point` |
| `msmtdate1` | Cast from String to DateType, renamed to `summit_date` |
| `deathdate` | Cast from String to DateType, renamed to `death_date` |
| `myear` | Renamed to `year` |
| `mseason` | Renamed to `season` |
| `msuccess` | Renamed to `success` |
| `mo2used` | Renamed to `oxygen_used` |
| `deathtype` | Renamed to `death_type` |
| `deathhgtm` | Renamed to `death_height` |
| `msmtterm` | Renamed to `summit_term` |
| 40+ columns | Dropped — not relevant to analysis (personal details, O2 details, route flags, duplicate dates, internal references) |
| Column order | Reordered logically by category |